In [1]:
import sys
sys.path.append("../")
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import torch

In [15]:
p_list_cm = [2**i for i in range(20, 28)]
base = matplotlib.colormaps["viridis"]
cmap  = matplotlib.colors.LinearSegmentedColormap.from_list(
    "viridis_trunc", base(np.linspace(0.05, 0.9, 256))
)
norm = matplotlib.colors.LogNorm(vmin=p_list_cm[0], vmax=p_list_cm[-1])

# Panels of figure 5

In [20]:
A = 4.9
gamma = 0.265

## 1st: T=128, GPT2 with APE

In [21]:
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "pdflatex",
    'font.family': 'serif',
    'text.usetex': True,
    'pgf.rcfonts': False,
    'font.size': 10,
    'lines.markersize': 4,
    'lines.linewidth': 1.0,
})

fig = plt.figure(figsize=(1*2.2, 1*2.2), dpi=120)
L, B, W, H = 0.18, 0.18, 0.72, 0.72 
ax  = fig.add_axes([L, B, W, H])

T = 128
depth = 12

results = torch.load(f"wikitext/learning-curves/ngrams_wikitext.BPE8192_T{T}_d{depth}.pt")
p_list = results['data']
print(p_list)
ngrams_list = [n.mean(dim=0) for n in results['ngrams']]

for i, p in enumerate(p_list):
    ax.plot([t+1 for t in range(T)], ngrams_list[i], color=cmap(norm(p+1)), label=fr'$P={p}$')

ax.plot([t+1 for t in range(11)], [A*(t+1)**(-gamma) for t in range(11)],'k--')

# ax.legend(fontsize=8, handlelength=1)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_ylabel(r'$\mathcal{L}_{n}$', fontsize=10)
ax.set_xlabel(r'$n$', fontsize=10)

ticks  = [ 3, 4, 6]
labels = [ "3", "4", "6"]
ax.set_yticks(ticks)
ax.set_yticklabels(labels)
ax.yaxis.set_minor_locator(mticker.NullLocator())

ax.set_ylim(2.3, 6.)

[1048576, 2097152, 4194304, 8388608, 16777216, 33554432, 67108864, 134217728]


(2.3, 6.0)

In [22]:
fig.savefig(f'wikitext/conditional-entropy_wikitext.BPE8192_T{T}_APE_Left.pdf', bbox_inches="tight")

## 2nd: T=128, GPT2 with RoPE

In [25]:
fig = plt.figure(constrained_layout=True, figsize=(1*2.1, 1*2.2), dpi=120)

fig = plt.figure(figsize=(1*2.2, 1*2.2), dpi=120)  # can be same for all
L, B, W, H = 0.18, 0.18, 0.72, 0.72 
ax  = fig.add_axes([L, B, W, H])                # <-- plot box is fixed

T = 128
depth = 12

results = torch.load(f"wikitext/learning-curves/ngrams_wikitext.BPE8192_T{T}_d{depth}_rope.pt")
p_list = results['data']
print(p_list)
ngrams_list = [n.mean(dim=0) for n in results['ngrams']]

for i, p in enumerate(p_list):
    ax.plot([t+1 for t in range(T)], ngrams_list[i], color=cmap(norm(p+1)), label=fr'$P={p}$')

ax.plot([t+1 for t in range(11)], [A*(t+1)**(-gamma) for t in range(11)],'k--')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$n$', fontsize=10)

ax.set_ylim(2.3, 6.)

ax.yaxis.set_minor_locator(mticker.NullLocator())
ax.yaxis.set_major_locator(mticker.NullLocator())


CB_W  = 0.04
CB_PAD = 0.02
want_cbar = True
sm = matplotlib.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array(np.array(p_list_cm))
if want_cbar:
    cax = fig.add_axes([L + W + CB_PAD, B, CB_W, H])
    cbar = fig.colorbar(sm, cax=cax)

cbar.ax.set_xlabel(r'$P$', labelpad=4)
cbar.ax.yaxis.set_label_position('right')
cbar.ax.yaxis.set_ticks_position('right')

ticks = [ 2**20, 2**22, 2**24, 2**26]
cbar.set_ticks(ticks)
exps = np.log2(ticks).astype(int)
cbar.set_ticklabels([rf"$2^{{{e}}}$" for e in exps])

[1048576, 2097152, 4194304, 8388608, 16777216, 33554432, 67108864, 134217728]


In [26]:
plt.savefig(f"wikitext/conditional-entropy_wikitext.BPE8192_T128_RoPE_Right.pdf", bbox_inches="tight")

## Final: Comparison at convergence

In [29]:
fig = plt.figure(figsize=(1*2.2, 1*2.2), dpi=120)
L, B, W, H = 0.18, 0.18, 0.72, 0.72 
ax  = fig.add_axes([L, B, W, H])

T = 128
depth = 12

# base = matplotlib.colormaps["plasma"]
# cmap  = matplotlib.colors.LinearSegmentedColormap.from_list(
#     "plasma_trunc", base(np.linspace(0.05, 0.9, 256))
# )
# norm = matplotlib.colors.LogNorm(vmin=p_list[0], vmax=p_list[-1])

colors = ["#8C1D18", "#1F4E79", "#6A8E3F", "#D89C1D", "#6B4C9A"]

# GPT2 APE
results = torch.load(f"wikitext/learning-curves/ngrams_wikitext.BPE8192_T{T}_d{depth}.pt")
p_list = results['data']
ngrams_list = [n.mean(dim=0) for n in results['ngrams']]
ax.plot([t+1 for t in range(T)], ngrams_list[-1], color=colors[0], label=fr'GPT-2 APE')

# GPT2 RoPE
results = torch.load(f"wikitext/learning-curves/ngrams_wikitext.BPE8192_T{T}_d{depth}_rope.pt")
p_list = results['data']
ngrams_list = [n.mean(dim=0) for n in results['ngrams']]
print(len(results["data"]), len(results["ngrams"]), results["ngrams"][0].shape, results["ngrams"][0].dtype)
ax.plot([t+1 for t in range(T)], ngrams_list[-1], color=colors[1], label=fr'GPT-2 RoPE')

# Gamma fit
ax.plot([t+1 for t in range(11)], [A*(t+1)**(-gamma) for t in range(11)],'k--', label=rf"$n^{{-\gamma}}$, $\gamma=${gamma}")

ax.legend(fontsize=7, handlelength=1)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$n$', fontsize=10)

ax.yaxis.set_minor_locator(mticker.NullLocator())
ax.yaxis.set_major_locator(mticker.NullLocator())

ax.set_ylim(2.3, 6.)

8 8 torch.Size([1, 128]) torch.float32


(2.3, 6.0)

In [30]:
fig.savefig(f'wikitext/conditional-entropy_wikitext.BPE8192_CompArch.pdf', bbox_inches="tight")